<a href="https://colab.research.google.com/github/Niarfe/DS6050_SPRING_2026_PROJECT/blob/Haejin/ISIC2019_EfficientNet_B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Download the ISIC2019 data and save in Drive

In [ ]:
!pip install timm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!wget -q "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip" -O "/content/train_imgs.zip"

In [ ]:
#Download ISIC 2019 data https://challenge.isic-archive.com/data/#2019 into /content/
!wget -q "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv" -O "/content/train_gt.csv"
!wget -q "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv" -O "/content/train_meta.csv"
!wget -q "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip" -O "/content/test_imgs.zip"
!wget -q "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv" -O "/content/test_gt.csv"


In [ ]:
#unzip from downloaded files in /content/
!unzip -q "/content/train_imgs.zip" -d /content/
!unzip -q "/content/test_imgs.zip" -d /content/

#verify the training/test/total counts
train_imgs = [f for f in os.listdir("/content/ISIC_2019_Training_Input") if f.endswith(".jpg")]
test_imgs  = [f for f in os.listdir("/content/ISIC_2019_Test_Input") if f.endswith(".jpg")]

print("Training images:", len(train_imgs))
print("Test images:",     len(test_imgs))
print("Total:",           len(train_imgs) + len(test_imgs))

unzip:  cannot find or open /content/train_imgs.zip, /content/train_imgs.zip.zip or /content/train_imgs.zip.ZIP.
unzip:  cannot find or open /content/test_imgs.zip, /content/test_imgs.zip.zip or /content/test_imgs.zip.ZIP.


NameError: name 'os' is not defined

In [ ]:
import json, time, warnings
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms

import timm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                              f1_score, confusion_matrix)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


# EfficientNet-B0

In [ ]:
# for Ablation studies
ARCHITECTURE = "efficientnet_b0"   # "resnet50" | "mobilenetv3_small"
LOSS_FN      = "weighted_ce"       # "ce" | "focal"
AUGMENTATION = "color"             # "none" | "geometric" | "strong"
PRETRAINED   = True                # False = train from scratch ablation
FREEZE_BB    = False               # True  = head-only training ablation

TRAIN_CSV  = "/content/train_gt.csv"
TEST_CSV   = "/content/test_gt.csv"
TRAIN_META = "/content/train_meta.csv"
TRAIN_DIR  = "/content/ISIC_2019_Training_Input"
TEST_DIR   = "/content/ISIC_2019_Test_Input"

CLASSES      = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES  = 8 #excluding the UNK class from the test set - TBD with team
IMG_SIZE     = 224 #standard ImageNet input size for efficientnet_b0 baseline first
BATCH_SIZE   = 32 #also in line with Prof feedback
EPOCHS       = 30 #same as above
LR           = 1e-4 #same as above
VAL_SPLIT    = 0.20 #same as above
PATIENCE     = 10 #same as above; early stopping (patience=10 on val BACC)
SEED         = 42 #just a default number. can be changed.

train_df_temp = pd.read_csv(TRAIN_CSV)
counts = train_df_temp[CLASSES].sum()
CLASS_FREQ = (counts / counts.sum()).to_dict()

print("Class frequencies:")
for c, f in CLASS_FREQ.items():
    print(f"  {c}: {f:.3f}")

print("Configuration set")

Class frequencies:
  MEL: 0.179
  NV: 0.508
  BCC: 0.131
  AK: 0.034
  BKL: 0.104
  DF: 0.009
  VASC: 0.010
  SCC: 0.025
Configuration set


LOSS_FN = "weighted_ce" in above code (recommended as default by Prof) shows class-weighted cross entropy with inverse, computed from the CSV file. For imabalance handling (ablation study), we will compare this against standard CE loss as well as focal loss and report per-class accuracy changes. "Hypothesis: weighted CE and focal loss improve minority-class accuracy by
+10–20% at the cost of↓2–3% on NV."

In [ ]:
#image preprocessing and augmentation
def pad_to_square(img):
    """Padding to square preserves lesion aspect ratio...don't squash"""
    w, h = img.size
    s = max(w, h)
    out = Image.new("RGB", (s, s), (0, 0, 0))
    out.paste(img, ((s - w) // 2, (s - h) // 2))
    return out

def get_transforms(split):
    base = [
        transforms.Lambda(pad_to_square),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ]
    if split != "train":
        return transforms.Compose(base)

    if AUGMENTATION == "none": #no augmentation
        aug = []
    elif AUGMENTATION == "geometric":
      #flip + rotation
        aug = [transforms.RandomHorizontalFlip(),
               transforms.RandomVerticalFlip(),
               transforms.RandomRotation(15)]
    elif AUGMENTATION == "color":
      #flip + roation + color changes(ColorJitter, RandomGrayscale)
        aug = [transforms.RandomHorizontalFlip(),
               transforms.RandomVerticalFlip(),
               transforms.RandomRotation(15),
               transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
               transforms.RandomGrayscale(p=0.05)]
    elif AUGMENTATION == "strong":
      #flip + rotation + color changes + RandAugment (or CutMix?)
        aug = [transforms.RandomHorizontalFlip(),
               transforms.RandomVerticalFlip(),
               transforms.RandomRotation(30),
               transforms.ColorJitter(0.3, 0.3, 0.3, 0.15),
               transforms.RandAugment(num_ops=2, magnitude=9)]
    return transforms.Compose(aug + base)

class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.img_dir}/{row['image']}.jpg").convert("RGB")
        if self.transform: img = self.transform(img)
        label = int(np.argmax([row[c] for c in CLASSES]))
        return img, label

def make_split(csv_path, meta_path):
  #using GroupShuffleSplit() to guarantee all images with the same lesion_id is in the same fold,
  #so no patient appears in both training and val..prevents the data leakage (Prof feedback).
    df   = pd.read_csv(csv_path)
    meta = pd.read_csv(meta_path)
    df   = df.merge(meta[["image","lesion_id"]], on="image", how="left")
    df["lesion_id"] = df["lesion_id"].fillna(df["image"]) #replace NAs in lesion_id w/ image

    print("Patient-grouped split using lesion_id")

    df["label"] = df[CLASSES].values.argmax(axis=1)

    gss = GroupShuffleSplit(1, test_size=VAL_SPLIT, random_state=SEED)
    ti, vi = next(gss.split(df, df["label"], groups=df["lesion_id"]))
    return df.iloc[ti].copy(), df.iloc[vi].copy()

# Build splits
train_df, val_df = make_split(TRAIN_CSV, TRAIN_META)
test_df = pd.read_csv(TEST_CSV)
if "UNK" in test_df.columns:
    test_df = test_df[test_df["UNK"] != 1].copy()

# ----------10% subset for quick testing --- To Be Removed-----
train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
val_df   = val_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
test_df  = test_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
print(f"Subset — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
# -----------to be removed ---------

# Build datasets
train_ds = ISICDataset(train_df, TRAIN_DIR, get_transforms("train"))
val_ds   = ISICDataset(val_df,   TRAIN_DIR, get_transforms("val"))
test_ds  = ISICDataset(test_df,  TEST_DIR,  get_transforms("test"))

# Build dataloaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Print class distribution
for name, df_ in [("TRAIN", train_df), ("VAL", val_df)]:
    labels = df_[CLASSES].values.argmax(axis=1)
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    print(f"\n{name} ({len(df_)} images):")

    for i, c in enumerate(CLASSES):
        print(f"  {c:6s}: {counts[i]:5d}  ({100*counts[i]/len(df_):4.1f}%)")

Patient-grouped split using lesion_id
Subset — train: 2015, val: 518, test: 619

TRAIN (2015 images):
  MEL   :   347  (17.2%)
  NV    :  1005  (49.9%)
  BCC   :   302  (15.0%)
  AK    :    64  ( 3.2%)
  BKL   :   197  ( 9.8%)
  DF    :    23  ( 1.1%)
  VASC  :    20  ( 1.0%)
  SCC   :    57  ( 2.8%)

VAL (518 images):
  MEL   :    97  (18.7%)
  NV    :   268  (51.7%)
  BCC   :    60  (11.6%)
  AK    :    26  ( 5.0%)
  BKL   :    54  (10.4%)
  DF    :     1  ( 0.2%)
  VASC  :     3  ( 0.6%)
  SCC   :     9  ( 1.7%)


Stratificiation in train and val worked.
Class imabalance is verey clear.

In [ ]:
#Implement EfficientNet-B0 baseline first

def build_model():
    model = timm.create_model("efficientnet_b0", pretrained=PRETRAINED, num_classes=NUM_CLASSES)
    if FREEZE_BB:
        for name, p in model.named_parameters():
            if not name.startswith("classifier"):
                p.requires_grad = False
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f"Backbone frozen — trainable: {trainable:,}/{total:,}")
    return model

def get_loss():
    freqs   = np.array([CLASS_FREQ[c] for c in CLASSES])
    weights = torch.tensor(1.0 / (NUM_CLASSES * freqs), dtype=torch.float32)
    weights = weights / weights.sum() * NUM_CLASSES
    if LOSS_FN == "ce":
        return nn.CrossEntropyLoss().to(device)
    elif LOSS_FN == "weighted_ce":
        return nn.CrossEntropyLoss(weight=weights.to(device))
    elif LOSS_FN == "focal":
        class FocalLoss(nn.Module):
            def __init__(self, gamma=2.0, alpha=None):
                super().__init__()
                self.gamma = gamma
                self.alpha = alpha
            def forward(self, logits, targets):
                ce = F.cross_entropy(logits, targets, reduction="none")
                pt = F.softmax(logits,1).gather(1,targets.unsqueeze(1)).squeeze(1)
                at = self.alpha.to(logits.device)[targets]
                return (at * (1-pt)**self.gamma * ce).mean()
        return FocalLoss(gamma=2.0, alpha=weights).to(device)

model     = build_model().to(device)
loss_fn   = get_loss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

total = sum(p.numel() for p in model.parameters())
print(f"Model: EfficientNet-B0 | Params: {total:,} | Device: {device}")
print(f"Loss: {LOSS_FN} | Pretrained: {PRETRAINED}")

Model: EfficientNet-B0 | Params: 4,017,796 | Device: cuda
Loss: weighted_ce | Pretrained: True


In [ ]:
#evaluation

@torch.no_grad() #turning off gradients tracking

def evaluate(model, loader):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    #inference time
    totla_time, total_imgs = 0.0, 0

    for imgs, labels in loader:
        imgs = imgs.to(device)

        t0 = time.time()
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        total_time += time.time() - t0
        total_imgs += imgs.size(0)

        all_labels.extend(labels.numpy())
        all_probs.extend(probs)
        all_preds.extend(probs.argmax(axis=1))

    ms_per_img = (total_time / total_imgs)*100

    labels = np.array(all_labels)
    probs  = np.array(all_probs)
    preds  = np.array(all_preds)

    bacc   = balanced_accuracy_score(labels, preds)
    f1_mac = f1_score(labels, preds, average="macro", zero_division=0)
    per_f1 = f1_score(labels, preds, average=None,
                      labels=list(range(NUM_CLASSES)), zero_division=0)

    aucs = {}
    for i, cls in enumerate(CLASSES):
        try:
            aucs[cls] = roc_auc_score((labels==i).astype(int), probs[:,i])
        except:
            aucs[cls] = float("nan")
    auc_mac = np.nanmean(list(aucs.values()))

    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = {}, {}

    for i, cls in enumerate(CLASSES):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        sens[cls] = tp/(tp+fn) if (tp+fn)>0 else 0.0
        spec[cls] = tn/(tn+fp) if (tn+fp)>0 else 0.0

    return {"bacc":bacc,
            "auc_macro":auc_mac,
            "f1_macro":f1_mac,
            "per_auc":aucs,
            "per_f1":dict(zip(CLASSES,per_f1)),
            "sensitivity":sens,
            "specificity":spec}

def print_metrics(m, split="val"):
    print(f"\n── {split.upper()} ──")
    print(f"  BACC: {m['bacc']:.4f}  |  AUC: {m['auc_macro']:.4f}  |  F1: {m['f1_macro']:.4f}")
    print(f"\n  {'Class':<8}{'AUC':>7}{'F1':>7}{'Sens':>7}{'Spec':>7}")
    for c in CLASSES:
        print(f"  {c:<8}{m['per_auc'][c]:>7.3f}{m['per_f1'][c]:>7.3f}"
              f"{m['sensitivity'][c]:>7.3f}{m['specificity'][c]:>7.3f}")

print("Metrics functions defined. Now, training loop for 30 epochs with early stopping!")

Metrics functions defined. Now, training loop for 30 epochs with early stopping!


In [ ]:
best_bacc, best_epoch, no_improve = -1, 0, 0
history = []
os.makedirs("/content/checkpoints", exist_ok=True)
ckpt_path = f"/content/checkpoints/efficientnet_b0_{LOSS_FN}_{AUGMENTATION}.pt"

print(f"Training EfficientNet-B0 | loss={LOSS_FN} | aug={AUGMENTATION} | pretrained={PRETRAINED}\n")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    scheduler.step()
    train_loss = run_loss / total
    train_acc  = correct / total

    # Validate
    m = evaluate(model, val_loader)
    if epoch % 5 == 0: #full metrics run every 5 epoch
        print_metrics(m, split="val")
    history.append({"epoch":epoch, "train_loss":train_loss, "train_acc":train_acc,
                    "val_bacc":m["bacc"], "val_auc":m["auc_macro"], "val_f1":m["f1_macro"]})

    print(f"Epoch {epoch:3d}/{EPOCHS} | loss={train_loss:.4f} | acc={train_acc:.4f} | "
          f"val BACC={m['bacc']:.4f} | AUC={m['auc_macro']:.4f} | F1={m['f1_macro']:.4f}", end="")

    if m["bacc"] > best_bacc:
        best_bacc, best_epoch = m["bacc"], epoch
        no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print("saved")
    else:
        no_improve += 1
        print(f"  (no improve {no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nDone in {(time.time()-t0)/60:.1f} min")
print(f"Best val BACC: {best_bacc:.4f} @ epoch {best_epoch}")

Training EfficientNet-B0 | loss=weighted_ce | aug=color | pretrained=True

Epoch   1/30 | loss=3.0313 | acc=0.1901 | val BACC=0.2711 | AUC=0.7252 | F1=0.2193saved
Epoch   2/30 | loss=1.9745 | acc=0.3871 | val BACC=0.3371 | AUC=0.7839 | F1=0.2591saved
Epoch   3/30 | loss=1.5032 | acc=0.4789 | val BACC=0.4583 | AUC=0.8137 | F1=0.2754saved
Epoch   4/30 | loss=1.4140 | acc=0.5122 | val BACC=0.4794 | AUC=0.8136 | F1=0.3026saved

── VAL ──
  BACC: 0.5115  |  AUC: 0.8400  |  F1: 0.3404

  Class       AUC     F1   Sens   Spec
  MEL       0.742  0.420  0.443  0.846
  NV        0.858  0.708  0.601  0.896
  BCC       0.880  0.490  0.617  0.882
  AK        0.787  0.148  0.154  0.951
  BKL       0.733  0.385  0.389  0.927
  DF        0.986  0.118  1.000  0.971
  VASC      0.989  0.333  0.667  0.986
  SCC       0.744  0.121  0.222  0.957
Epoch   5/30 | loss=1.0040 | acc=0.5826 | val BACC=0.5115 | AUC=0.8400 | F1=0.3404saved
Epoch   6/30 | loss=0.9144 | acc=0.6069 | val BACC=0.5307 | AUC=0.8355 | F1=